
# Eye ROI Deepfake Detection — Xception (SSOT v2.0)

Bu notebook, yüklenen **Mouth ROI / Xception** deneyinin **Kader → Deney 1 → Göz** veri yapısına uyarlanmış sürümüdür.

Temel kararlar:

- Model girdisi: `eye_roi_output/{real,fake}/{train,val,test}/combined/**`
- Pozitif sınıf: `FAKE = 1`, negatif sınıf: `REAL = 0`
- Model: `timm` içindeki ImageNet ön-eğitimli `legacy_xception`
- Eğitim: önce yalnızca classifier head, sonra düşük öğrenme oranıyla tüm model fine-tuning
- Test kümesi model/threshold seçimi için kullanılmaz
- Threshold yalnızca validation kümesinden seçilir
- Çıktılar yalnızca `Kader/Deney 1/Sonuçlar/<run_id>/` altına yazılır
- Run ID: `YYYYMMDD_HHMM_eye_xception_seed42`
- Metadata, split leakage audit, smoke test, NaN/Inf kontrolü, atomik checkpoint, resume, test inference ve İngilizce rapor görselleri dahildir.

> **Önemli:** Split güvenliği için `source_video` dosya adından çıkarılır. Dosya adı video kimliğini açık biçimde taşımıyorsa kod tahmin yürütmez; örnek dosya adlarını göstererek durur ve `source_video_regex` ayarının verilmesini ister.


In [1]:

# ============================================================
# CELL 1 — DEPENDENCIES, IMPORTS, REPRODUCIBILITY
# ============================================================

!pip -q install "timm>=1.0,<2.0" "PyYAML>=6.0,<7.0"

import os
import re
import gc
import json
import math
import time
import random
import shutil
import hashlib
import platform
import subprocess
from datetime import datetime
from pathlib import Path
from collections import Counter

import yaml
import numpy as np
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from tqdm.auto import tqdm

SEED = 42

def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # Reproducibility > raw speed for this experiment.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_global_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"

print("=" * 78)
print("EYE ROI XCEPTION — ENVIRONMENT")
print("=" * 78)
print("Python :", platform.python_version())
print("PyTorch:", torch.__version__)
print("timm   :", timm.__version__)
print("Device :", DEVICE)
print("Seed   :", SEED)


EYE ROI XCEPTION — ENVIRONMENT
Python : 3.12.13
PyTorch: 2.11.0+cu128
timm   : 1.0.28
Device : cuda
Seed   : 42


In [2]:

# ============================================================
# CELL 2 — GOOGLE DRIVE, STRICT PATH RESOLUTION, YAML CONFIG
# ============================================================

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

MYDRIVE = Path("/content/drive/MyDrive")

def find_unique_child(parent: Path, accepted_names):
    """
    Unicode-normalization kaynaklı klasör adı farklarını sessizce tahmin etmek
    yerine mevcut çocuk klasörleri normalize ederek tek eşleşme arar.
    """
    import unicodedata

    if not parent.exists():
        raise FileNotFoundError(f"Parent directory does not exist: {parent}")

    def norm(s: str) -> str:
        s = unicodedata.normalize("NFKD", s)
        s = "".join(ch for ch in s if not unicodedata.combining(ch))
        return s.casefold().strip()

    wanted = {norm(x) for x in accepted_names}
    matches = [p for p in parent.iterdir() if p.is_dir() and norm(p.name) in wanted]

    if len(matches) == 1:
        return matches[0]

    if not matches:
        existing = [p.name for p in parent.iterdir() if p.is_dir()]
        raise FileNotFoundError(
            f"Expected one of {accepted_names} under {parent}.\n"
            f"Existing directories: {existing[:50]}"
        )

    raise RuntimeError(f"Ambiguous directory match under {parent}: {matches}")


AISC_ROOT = find_unique_child(
    MYDRIVE,
    [
        "AISC DeepFake Çalışmaları",
        "AISC Deepfake Çalışmaları",
        "AISC Çalışmalar",
        "AISC Çalışmalar",
    ],
)

DENEYLER_ROOT = find_unique_child(AISC_ROOT, ["Deneyler"])
KADER_ROOT = find_unique_child(DENEYLER_ROOT, ["Kader"])
DENEY1_ROOT = find_unique_child(KADER_ROOT, ["Deney 1", "Deney1"])
EYE_ROOT = find_unique_child(DENEY1_ROOT, ["Göz", "Goz"])
EYE_ROI_ROOT = find_unique_child(EYE_ROOT, ["eye_roi_output"])
FINAL_RESULTS_ROOT = find_unique_child(DENEY1_ROOT, ["Sonuçlar", "Sonuclar"])

CONFIG_DIR = DENEY1_ROOT / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = CONFIG_DIR / "experiment_eye_xception.yaml"

DEFAULT_CONFIG = {
    "seed": 42,
    "region": "eye",
    "roi_variant": "combined",
    "model_name": "legacy_xception",
    "pretrained": True,
    "image_size": 299,
    "batch_size": 16,
    "num_workers": 2,
    "initial_epochs": 12,
    "finetune_epochs": 8,
    "initial_lr": 1.0e-4,
    "finetune_lr": 1.0e-5,
    "weight_decay": 1.0e-4,
    "early_stopping_patience": 4,
    "scheduler_patience": 2,
    "scheduler_factor": 0.5,
    "gradient_clip_norm": 5.0,
    "checkpoint_keep_last_n": 3,
    "threshold_grid_points": 181,
    "compute_sha256": True,
    "verify_image_files": True,
    "source_video_regex": None,
    "strict_source_video": True,
    "resume_active_run": True,
    "train_augmentation": {
        "horizontal_flip_p": 0.5,
        "rotation_deg": 5,
        "brightness": 0.08,
        "contrast": 0.08,
    },
}

if not CONFIG_PATH.exists():
    tmp = CONFIG_PATH.with_suffix(".yaml.tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        yaml.safe_dump(DEFAULT_CONFIG, f, sort_keys=False, allow_unicode=True)
    os.replace(tmp, CONFIG_PATH)

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)

required_config_keys = set(DEFAULT_CONFIG)
missing_cfg = required_config_keys.difference(CONFIG)
if missing_cfg:
    raise KeyError(f"Missing config keys: {sorted(missing_cfg)}")

if int(CONFIG["seed"]) != SEED:
    SEED = int(CONFIG["seed"])
    set_global_seed(SEED)

ACTIVE_RUN_FILE = DENEY1_ROOT / ".active_eye_xception_run_id.txt"

if CONFIG["resume_active_run"] and ACTIVE_RUN_FILE.exists():
    RUN_ID = ACTIVE_RUN_FILE.read_text(encoding="utf-8").strip()
else:
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M") + f"_eye_xception_seed{SEED}"
    tmp = ACTIVE_RUN_FILE.with_suffix(".tmp")
    tmp.write_text(RUN_ID, encoding="utf-8")
    os.replace(tmp, ACTIVE_RUN_FILE)

OUTPUT_ROOT = FINAL_RESULTS_ROOT / RUN_ID
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
LOG_DIR = OUTPUT_ROOT / "logs"
METRICS_DIR = OUTPUT_ROOT / "metrics"
PREDICTIONS_DIR = OUTPUT_ROOT / "predictions"
FIGURES_DIR = OUTPUT_ROOT / "figures"
METADATA_DIR = OUTPUT_ROOT / "metadata"

for d in [
    OUTPUT_ROOT,
    CHECKPOINT_DIR,
    LOG_DIR,
    METRICS_DIR,
    PREDICTIONS_DIR,
    FIGURES_DIR,
    METADATA_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

RESOLVED_CONFIG_PATH = OUTPUT_ROOT / "config_resolved.yaml"
with open(RESOLVED_CONFIG_PATH.with_suffix(".yaml.tmp"), "w", encoding="utf-8") as f:
    yaml.safe_dump(CONFIG, f, sort_keys=False, allow_unicode=True)
os.replace(RESOLVED_CONFIG_PATH.with_suffix(".yaml.tmp"), RESOLVED_CONFIG_PATH)

print("=" * 78)
print("RESOLVED PROJECT PATHS")
print("=" * 78)
print("AISC root       :", AISC_ROOT)
print("Kader / Deney 1 :", DENEY1_ROOT)
print("Eye ROI root    :", EYE_ROI_ROOT)
print("Results root    :", FINAL_RESULTS_ROOT)
print("Run ID          :", RUN_ID)
print("Output root     :", OUTPUT_ROOT)


Mounted at /content/drive
RESOLVED PROJECT PATHS
AISC root       : /content/drive/MyDrive/AISC DeepFake Çalışmaları
Kader / Deney 1 : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1
Eye ROI root    : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output
Results root    : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar
Run ID          : 20260807_1004_eye_xception_seed42
Output root     : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/20260807_1004_eye_xception_seed42


In [3]:

# ============================================================
# CELL 3 — ATOMIC I/O, CHECKPOINT, HASHING, ENVIRONMENT LOCK
# ============================================================

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def atomic_json_dump(payload, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_suffix(target.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    with open(tmp, "r", encoding="utf-8") as f:
        json.load(f)  # integrity validation
    os.replace(tmp, target)

def atomic_csv_dump(df: pd.DataFrame, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_suffix(target.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    check = pd.read_csv(tmp)
    if len(check) != len(df):
        raise RuntimeError(f"CSV integrity check failed: {target}")
    os.replace(tmp, target)

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def stable_sample_id(label: str, split: str, relative_path: str) -> str:
    raw = f"{label}|{split}|{relative_path}".encode("utf-8")
    return hashlib.sha256(raw).hexdigest()[:16]

def capture_rng_state():
    state = {
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda_rng_state"] = torch.cuda.get_rng_state_all()
    else:
        state["cuda_rng_state"] = None
    return state

def restore_rng_state(state):
    random.setstate(state["python_rng_state"])
    np.random.set_state(state["numpy_rng_state"])
    torch.set_rng_state(state["torch_rng_state"])
    if torch.cuda.is_available() and state.get("cuda_rng_state") is not None:
        torch.cuda.set_rng_state_all(state["cuda_rng_state"])

def atomic_save_checkpoint(state: dict, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_suffix(target.suffix + ".tmp")

    torch.save(state, tmp)

    loaded = torch.load(tmp, map_location="cpu", weights_only=False)
    required_keys = {
        "epoch",
        "stage",
        "model_state_dict",
        "optimizer_state_dict",
        "scheduler_state_dict",
        "scaler_state_dict",
        "best_metric_score",
        "config",
        "rng_state",
    }
    if not required_keys.issubset(loaded):
        missing = required_keys.difference(loaded)
        raise RuntimeError(f"Checkpoint validation failed; missing keys: {sorted(missing)}")

    os.replace(tmp, target)

def rotate_epoch_checkpoints(stage_dir: Path, keep_last_n: int) -> None:
    epoch_files = sorted(
        stage_dir.glob("epoch_*.ckpt"),
        key=lambda p: int(re.search(r"epoch_(\d+)", p.stem).group(1)),
    )
    for old in epoch_files[:-keep_last_n]:
        old.unlink()

# Lock actual environment versions for reproducibility.
try:
    freeze = subprocess.check_output(
        ["python", "-m", "pip", "freeze"],
        text=True,
    )
    req_tmp = OUTPUT_ROOT / "requirements_lock.txt.tmp"
    req_tmp.write_text(freeze, encoding="utf-8")
    os.replace(req_tmp, OUTPUT_ROOT / "requirements_lock.txt")
except Exception as e:
    # The standard forbids silently ignoring write failures.
    raise RuntimeError("Could not save requirements lock.") from e

environment = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "timm": timm.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "device": str(DEVICE),
    "seed": SEED,
}
atomic_json_dump(environment, OUTPUT_ROOT / "environment.json")

print("Atomic I/O utilities ready.")


Atomic I/O utilities ready.


In [5]:
# ============================================================
# CELL 4 — BUILD AUDITABLE EYE METADATA / MANIFEST
# ============================================================

def discover_images(folder: Path):
    """
    Recursively discovers supported image files under a dataset folder.
    """
    if not folder.exists():
        raise FileNotFoundError(
            f"Dataset directory missing: {folder}"
        )

    return sorted(
        p
        for p in folder.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    )


def parse_eye_roi_filename(path: Path):
    """
    Parse Eye ROI filenames without inventing source-video information.

    Expected examples
    -----------------
    real_train_00000__face_00.jpg
    real_train_00011__face_01.jpg
    fake_train_00125__face_00.jpg
    real_val_00042__face_00.jpg
    fake_test_00007__face_02.jpg

    Important
    ---------
    The numeric field is interpreted as frame/sample index.

    It is NOT automatically interpreted as source_video because the
    filename does not provide enough evidence to prove that.
    """

    stem = path.stem

    pattern = re.compile(
        r"^(?P<label>real|fake)_"
        r"(?P<split>train|val|test)_"
        r"(?P<frame_index>\d+)"
        r"__face_(?P<face_index>\d+)$",
        flags=re.IGNORECASE,
    )

    match = pattern.match(stem)

    if match is None:
        return None

    return {
        "parsed_label": match.group("label").lower(),
        "parsed_split": match.group("split").lower(),
        "frame_index": int(match.group("frame_index")),
        "face_index": int(match.group("face_index")),
    }


# ============================================================
# DATASET DIRECTORY DEFINITIONS
# ============================================================

dataset_folders = {}

for label in ["real", "fake"]:
    for split in ["train", "val", "test"]:

        dataset_folder = (
            EYE_ROI_ROOT
            / label
            / split
            / CONFIG["roi_variant"]
        )

        dataset_folders[(label, split)] = dataset_folder


# ============================================================
# DIRECTORY EXISTENCE QUALITY GATE
# ============================================================

print("=" * 78)
print("EYE ROI DATASET DIRECTORY CHECK")
print("=" * 78)

for (label, split), folder in dataset_folders.items():

    if not folder.exists():
        raise FileNotFoundError(
            f"Missing required Eye ROI directory:\n"
            f"label = {label}\n"
            f"split = {split}\n"
            f"path  = {folder}"
        )

    print(
        f"{label.upper():<4} "
        f"{split:<5} -> {folder}"
    )


# ============================================================
# METADATA CONSTRUCTION
# ============================================================

rows = []

total_inputs = 0
success_count = 0
skipped_count = 0
error_count = 0

filename_schema_errors = []


print()
print("=" * 78)
print("BUILDING AUDITABLE EYE ROI METADATA")
print("=" * 78)


for (label, split), folder in dataset_folders.items():

    files = discover_images(folder)

    print(
        f"{label.upper():<4} "
        f"{split:<5}: "
        f"{len(files):>7} images"
    )

    for path in tqdm(
        files,
        desc=f"{label}-{split}",
        leave=False,
    ):

        total_inputs += 1

        relative_path = (
            path
            .relative_to(EYE_ROI_ROOT)
            .as_posix()
        )

        # ----------------------------------------------------
        # DEFAULT VALUES
        # ----------------------------------------------------

        status = "SUCCESS"
        skip_reason = ""

        frame_index = -1
        face_index = -1

        content_hash = ""

        parsed = None


        try:

            # =================================================
            # 1. FILENAME SCHEMA VALIDATION
            # =================================================

            parsed = parse_eye_roi_filename(path)

            if parsed is None:

                filename_schema_errors.append(
                    relative_path
                )

                raise ValueError(
                    "Filename does not match expected Eye ROI schema. "
                    "Expected format: "
                    "<real|fake>_<train|val|test>_"
                    "<frame_index>__face_<face_index>.<ext>"
                )


            frame_index = parsed["frame_index"]
            face_index = parsed["face_index"]


            # =================================================
            # 2. LABEL CONSISTENCY CHECK
            # =================================================

            if parsed["parsed_label"] != label:

                raise ValueError(
                    "Filename/directory label mismatch: "
                    f"filename={parsed['parsed_label']} "
                    f"directory={label}"
                )


            # =================================================
            # 3. SPLIT CONSISTENCY CHECK
            # =================================================

            if parsed["parsed_split"] != split:

                raise ValueError(
                    "Filename/directory split mismatch: "
                    f"filename={parsed['parsed_split']} "
                    f"directory={split}"
                )


            # =================================================
            # 4. IMAGE FILE INTEGRITY CHECK
            # =================================================

            if CONFIG["verify_image_files"]:

                with Image.open(path) as image:

                    image.verify()


            # =================================================
            # 5. SHA256 CONTENT HASH
            # =================================================

            if CONFIG["compute_sha256"]:

                content_hash = sha256_file(
                    path
                )


            success_count += 1


        except Exception as exc:

            status = "ERROR"

            skip_reason = (
                f"{type(exc).__name__}: "
                f"{str(exc)[:300]}"
            )

            error_count += 1


        # =====================================================
        # STABLE SAMPLE ID
        # =====================================================

        sample_id = stable_sample_id(
            label,
            split,
            relative_path,
        )


        # =====================================================
        # SOURCE FRAME IDENTIFIER
        # =====================================================

        if frame_index >= 0:

            source_frame = (
                f"{label}_{split}_"
                f"{frame_index:05d}"
            )

        else:

            source_frame = None


        # =====================================================
        # IMPORTANT:
        #
        # source_video IS NOT FABRICATED HERE.
        #
        # The filename:
        #
        # real_train_00000__face_00.jpg
        #
        # proves frame/sample index and face index,
        # but does not by itself prove original source video ID.
        #
        # source_video must later be restored from authoritative
        # ROI/source metadata before video-level leakage audit.
        # =====================================================

        rows.append(
            {
                "sample_id": sample_id,

                "source_video": None,

                "source_frame": source_frame,

                "frame_index": int(frame_index),

                "face_index": int(face_index),

                "roi_state": "combined_eye",

                "label": label,

                "split": split,

                "status": status,

                "skip_reason": skip_reason,

                "sha256": content_hash,

                "output_path": str(path),

                "relative_path": relative_path,

                "run_id": RUN_ID,
            }
        )


# ============================================================
# CREATE DATAFRAME
# ============================================================

metadata = pd.DataFrame(rows)


# ============================================================
# METADATA SCHEMA QUALITY GATE
# ============================================================

REQUIRED_METADATA_COLUMNS = {
    "sample_id",
    "source_video",
    "source_frame",
    "frame_index",
    "face_index",
    "roi_state",
    "label",
    "split",
    "status",
    "skip_reason",
    "sha256",
    "output_path",
    "relative_path",
    "run_id",
}


missing_columns = (
    REQUIRED_METADATA_COLUMNS
    .difference(metadata.columns)
)


if missing_columns:

    raise AssertionError(
        "Metadata schema is incomplete. "
        f"Missing columns: "
        f"{sorted(missing_columns)}"
    )


# ============================================================
# DATA ACCOUNTING QUALITY GATE
# ============================================================

assert (
    total_inputs
    ==
    success_count
    +
    skipped_count
    +
    error_count
), (
    "Input accounting mismatch: "
    f"total={total_inputs}, "
    f"success={success_count}, "
    f"skipped={skipped_count}, "
    f"errors={error_count}"
)


# ============================================================
# UNIQUE SAMPLE ID QUALITY GATE
# ============================================================

assert metadata["sample_id"].is_unique, (
    "Duplicate sample_id detected."
)


# ============================================================
# OUTPUT PATH QUALITY GATE
# ============================================================

assert metadata["output_path"].notna().all(), (
    "Missing output_path detected."
)


# ============================================================
# SUCCESS FILE EXISTENCE QUALITY GATE
# ============================================================

success_paths = metadata.loc[
    metadata["status"] == "SUCCESS",
    "output_path",
]


missing_success_files = [
    p
    for p in success_paths
    if not Path(p).exists()
]


if missing_success_files:

    raise FileNotFoundError(
        "Some SUCCESS files do not exist on disk.\n"
        "Examples:\n- "
        +
        "\n- ".join(
            missing_success_files[:20]
        )
    )


# ============================================================
# LABEL QUALITY GATE
# ============================================================

allowed_labels = {
    "real",
    "fake",
}


invalid_labels = (
    set(metadata["label"].unique())
    -
    allowed_labels
)


if invalid_labels:

    raise AssertionError(
        f"Invalid labels detected: "
        f"{sorted(invalid_labels)}"
    )


# ============================================================
# SPLIT QUALITY GATE
# ============================================================

allowed_splits = {
    "train",
    "val",
    "test",
}


invalid_splits = (
    set(metadata["split"].unique())
    -
    allowed_splits
)


if invalid_splits:

    raise AssertionError(
        f"Invalid splits detected: "
        f"{sorted(invalid_splits)}"
    )


# ============================================================
# FRAME INDEX QUALITY GATE
# ============================================================

successful_metadata = metadata.loc[
    metadata["status"] == "SUCCESS"
]


assert (
    successful_metadata["frame_index"] >= 0
).all(), (
    "Negative/invalid frame_index detected "
    "among SUCCESS samples."
)


assert (
    successful_metadata["face_index"] >= 0
).all(), (
    "Negative/invalid face_index detected "
    "among SUCCESS samples."
)


# ============================================================
# EXACT SAMPLE DUPLICATE CHECK
# ============================================================

duplicate_identity = (
    successful_metadata
    .duplicated(
        subset=[
            "label",
            "split",
            "frame_index",
            "face_index",
            "relative_path",
        ],
        keep=False,
    )
)


if duplicate_identity.any():

    duplicate_preview = (
        successful_metadata.loc[
            duplicate_identity,
            [
                "label",
                "split",
                "frame_index",
                "face_index",
                "relative_path",
            ],
        ]
        .head(20)
    )

    raise AssertionError(
        "Duplicate Eye ROI identities detected.\n\n"
        +
        duplicate_preview.to_string(
            index=False
        )
    )


# ============================================================
# IMAGE HASH DUPLICATE CHECK
# ============================================================

if CONFIG["compute_sha256"]:

    empty_hashes = (
        successful_metadata["sha256"]
        .astype(str)
        .str.strip()
        .eq("")
    )

    if empty_hashes.any():

        raise AssertionError(
            "Missing SHA256 value among "
            "SUCCESS samples."
        )


# ============================================================
# FAIL ON CORRUPT / INVALID FILES
# ============================================================

if error_count > 0:

    error_preview = (
        metadata.loc[
            metadata["status"] == "ERROR",
            [
                "relative_path",
                "skip_reason",
            ],
        ]
        .head(20)
    )

    print()
    print("=" * 78)
    print("ERROR PREVIEW")
    print("=" * 78)

    print(
        error_preview.to_string(
            index=False
        )
    )

    raise RuntimeError(
        f"{error_count} invalid or unreadable "
        "Eye ROI files detected. "
        "Metadata build stopped."
    )


# ============================================================
# SAVE METADATA
# ============================================================

METADATA_PATH = (
    METADATA_DIR
    /
    "eye_combined_metadata.csv"
)


atomic_csv_dump(
    metadata,
    METADATA_PATH,
)


# ============================================================
# DATA ACCOUNTING REPORT
# ============================================================

accounting = {
    "total_inputs": int(
        total_inputs
    ),

    "success_count": int(
        success_count
    ),

    "skipped_count": int(
        skipped_count
    ),

    "error_count": int(
        error_count
    ),

    "real_train": int(
        (
            (metadata["label"] == "real")
            &
            (metadata["split"] == "train")
            &
            (metadata["status"] == "SUCCESS")
        ).sum()
    ),

    "real_val": int(
        (
            (metadata["label"] == "real")
            &
            (metadata["split"] == "val")
            &
            (metadata["status"] == "SUCCESS")
        ).sum()
    ),

    "real_test": int(
        (
            (metadata["label"] == "real")
            &
            (metadata["split"] == "test")
            &
            (metadata["status"] == "SUCCESS")
        ).sum()
    ),

    "fake_train": int(
        (
            (metadata["label"] == "fake")
            &
            (metadata["split"] == "train")
            &
            (metadata["status"] == "SUCCESS")
        ).sum()
    ),

    "fake_val": int(
        (
            (metadata["label"] == "fake")
            &
            (metadata["split"] == "val")
            &
            (metadata["status"] == "SUCCESS")
        ).sum()
    ),

    "fake_test": int(
        (
            (metadata["label"] == "fake")
            &
            (metadata["split"] == "test")
            &
            (metadata["status"] == "SUCCESS")
        ).sum()
    ),
}


ACCOUNTING_PATH = (
    METADATA_DIR
    /
    "data_accounting.json"
)


atomic_json_dump(
    accounting,
    ACCOUNTING_PATH,
)


# ============================================================
# SPLIT SAMPLE COUNT TABLE
# ============================================================

split_sample_counts = (
    metadata.loc[
        metadata["status"] == "SUCCESS"
    ]
    .groupby(
        [
            "split",
            "label",
        ]
    )
    .size()
    .rename("sample_count")
    .reset_index()
)


SPLIT_SAMPLE_COUNT_PATH = (
    METADATA_DIR
    /
    "split_sample_counts.csv"
)


atomic_csv_dump(
    split_sample_counts,
    SPLIT_SAMPLE_COUNT_PATH,
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 78)
print("EYE ROI METADATA BUILD COMPLETED")
print("=" * 78)

print(
    f"Total inputs : "
    f"{total_inputs:,}"
)

print(
    f"SUCCESS      : "
    f"{success_count:,}"
)

print(
    f"SKIPPED      : "
    f"{skipped_count:,}"
)

print(
    f"ERROR        : "
    f"{error_count:,}"
)

print()

print(
    split_sample_counts.to_string(
        index=False
    )
)

print()

print(
    "Metadata saved:"
)

print(
    METADATA_PATH
)

print()

print(
    "Accounting saved:"
)

print(
    ACCOUNTING_PATH
)

print()

print(
    "IMPORTANT:"
)

print(
    "source_video has intentionally NOT been inferred "
    "from filenames such as "
    "'real_train_00000__face_00.jpg'."
)

print(
    "The numeric identifier is stored as frame_index. "
    "Original source_video information must be joined "
    "from authoritative ROI/source metadata before the "
    "video-level split leakage quality gate."
)

EYE ROI DATASET DIRECTORY CHECK
REAL train -> /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output/real/train/combined
REAL val   -> /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output/real/val/combined
REAL test  -> /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output/real/test/combined
FAKE train -> /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output/fake/train/combined
FAKE val   -> /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output/fake/val/combined
FAKE test  -> /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output/fake/test/combined

BUILDING AUDITABLE EYE ROI METADATA
REAL train:    1197 images


real-train:   0%|          | 0/1197 [00:00<?, ?it/s]

REAL val  :     155 images


real-val:   0%|          | 0/155 [00:00<?, ?it/s]

REAL test :     146 images


...

FAKE val  :     141 images


fake-val:   0%|          | 0/141 [00:00<?, ?it/s]

FAKE test :     156 images


fake-test:   0%|          | 0/156 [00:00<?, ?it/s]


EYE ROI METADATA BUILD COMPLETED
Total inputs : 2,986
SUCCESS      : 2,986
SKIPPED      : 0
ERROR        : 0

split label  sample_count
 test  fake           156
 test  real           146
train  fake          1191
train  real          1197
  val  fake           141
  val  real           155

Metadata saved:
/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/20260807_1004_eye_xception_seed42/metadata/eye_combined_metadata.csv

Accounting saved:
/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/20260807_1004_eye_xception_seed42/metadata/data_accounting.json

IMPORTANT:
source_video has intentionally NOT been inferred from filenames such as 'real_train_00000__face_00.jpg'.
The numeric identifier is stored as frame_index. Original source_video information must be joined from authoritative ROI/source metadata before the video-level split leakage quality gate.


In [7]:
# ============================================================
# CELL 5 — SPLIT / LEAKAGE QUALITY GATE
# ============================================================

success_meta = metadata.loc[
    metadata["status"] == "SUCCESS"
].copy()


# ============================================================
# 1. METADATA SCHEMA QUALITY GATE
# ============================================================

required_columns = {
    "sample_id",
    "source_video",
    "source_frame",
    "frame_index",
    "face_index",
    "label",
    "split",
    "status",
    "sha256",
    "output_path",
    "run_id",
}

missing_columns = required_columns.difference(
    success_meta.columns
)

if missing_columns:
    raise AssertionError(
        "Metadata schema missing required columns: "
        f"{sorted(missing_columns)}"
    )


if success_meta.empty:
    raise AssertionError(
        "No SUCCESS samples available for leakage audit."
    )


# ============================================================
# 2. BASIC SPLIT VALIDATION
# ============================================================

allowed_splits = {
    "train",
    "val",
    "test",
}

found_splits = set(
    success_meta["split"]
    .astype(str)
    .str.lower()
    .unique()
)

unexpected_splits = (
    found_splits
    -
    allowed_splits
)

if unexpected_splits:
    raise AssertionError(
        "Unexpected split values detected: "
        f"{sorted(unexpected_splits)}"
    )


missing_splits = (
    allowed_splits
    -
    found_splits
)

if missing_splits:
    raise AssertionError(
        "Required split(s) missing: "
        f"{sorted(missing_splits)}"
    )


# ============================================================
# 3. SAMPLE-ID LEAKAGE CHECK
# ============================================================

train_sample_ids = set(
    success_meta.loc[
        success_meta["split"] == "train",
        "sample_id",
    ]
)

val_sample_ids = set(
    success_meta.loc[
        success_meta["split"] == "val",
        "sample_id",
    ]
)

test_sample_ids = set(
    success_meta.loc[
        success_meta["split"] == "test",
        "sample_id",
    ]
)


assert train_sample_ids.isdisjoint(
    val_sample_ids
), (
    "Train/Val sample_id leakage detected."
)


assert train_sample_ids.isdisjoint(
    test_sample_ids
), (
    "Train/Test sample_id leakage detected."
)


assert val_sample_ids.isdisjoint(
    test_sample_ids
), (
    "Val/Test sample_id leakage detected."
)


print(
    "[PASS] sample_id split isolation"
)


# ============================================================
# 4. EXACT CONTENT HASH LEAKAGE CHECK
# ============================================================

HASH_LEAKAGE_STATUS = "NOT_CHECKED"

cross_split_hashes = pd.Series(
    dtype="int64"
)


if CONFIG["compute_sha256"]:

    empty_hash_mask = (
        success_meta["sha256"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    )

    if empty_hash_mask.any():

        raise AssertionError(
            "SHA256 leakage audit requested, "
            "but some SUCCESS samples have empty hashes."
        )


    hash_split_counts = (
        success_meta
        .groupby("sha256")["split"]
        .nunique()
    )


    cross_split_hashes = (
        hash_split_counts[
            hash_split_counts > 1
        ]
    )


    if len(cross_split_hashes) > 0:

        leaking_hash_values = set(
            cross_split_hashes.index
        )

        leak_preview = (
            success_meta.loc[
                success_meta["sha256"].isin(
                    leaking_hash_values
                ),
                [
                    "split",
                    "label",
                    "relative_path",
                    "sha256",
                ],
            ]
            .sort_values(
                [
                    "sha256",
                    "split",
                ]
            )
            .head(30)
        )


        print()
        print("=" * 78)
        print("EXACT CONTENT LEAKAGE PREVIEW")
        print("=" * 78)

        print(
            leak_preview.to_string(
                index=False
            )
        )


        raise AssertionError(
            "Exact duplicate image content crosses "
            f"Train/Val/Test boundaries: "
            f"{len(cross_split_hashes)} SHA256 hashes."
        )


    HASH_LEAKAGE_STATUS = "PASSED"

    print(
        "[PASS] exact SHA256 content isolation"
    )


else:

    HASH_LEAKAGE_STATUS = "DISABLED"

    print(
        "[WARNING] SHA256 duplicate leakage audit "
        "is disabled in CONFIG."
    )


# ============================================================
# 5. SOURCE-FRAME IDENTITY CHECK
# ============================================================

#
# source_frame here means:
#
#   real_train_00000
#   fake_val_00020
#
# It is useful for duplicate accounting,
# but it MUST NOT be treated as source_video.
#

source_frame_duplicates = (
    success_meta
    .groupby(
        [
            "label",
            "split",
            "source_frame",
            "face_index",
        ]
    )
    .size()
)


bad_source_frame_duplicates = (
    source_frame_duplicates[
        source_frame_duplicates > 1
    ]
)


if len(bad_source_frame_duplicates) > 0:

    raise AssertionError(
        "Duplicate source_frame/face identities "
        f"detected: "
        f"{len(bad_source_frame_duplicates)}"
    )


print(
    "[PASS] source-frame / face identity uniqueness"
)


# ============================================================
# 6. VIDEO-LEVEL LEAKAGE CHECK
# ============================================================

#
# CRITICAL:
#
# We only perform video-level isdisjoint assertions when
# source_video is actually known.
#
# We DO NOT manufacture a video ID from frame_index.
#

source_video_series = (
    success_meta["source_video"]
    .replace(
        {
            "": np.nan,
            "None": np.nan,
            "nan": np.nan,
        }
    )
)


source_video_available_mask = (
    source_video_series.notna()
)


known_video_count = int(
    source_video_available_mask.sum()
)

total_success = int(
    len(success_meta)
)


VIDEO_LEAKAGE_STATUS = (
    "NOT_AVAILABLE"
)

VIDEO_LEAKAGE_REASON = (
    "source_video is not present in the current Eye ROI filenames. "
    "Video-level leakage cannot be scientifically verified from "
    "these filenames alone."
)


video_counts = None


if known_video_count == total_success:

    success_meta["source_video"] = (
        source_video_series
        .astype(str)
        .str.strip()
    )


    # Namespace by label because a REAL video ID and a FAKE
    # video ID could theoretically use the same numeric name.
    success_meta["video_key"] = (
        success_meta["label"]
        .astype(str)
        .str.lower()
        +
        "::"
        +
        success_meta["source_video"]
        .astype(str)
    )


    train_video_ids = set(
        success_meta.loc[
            success_meta["split"] == "train",
            "video_key",
        ]
    )


    val_video_ids = set(
        success_meta.loc[
            success_meta["split"] == "val",
            "video_key",
        ]
    )


    test_video_ids = set(
        success_meta.loc[
            success_meta["split"] == "test",
            "video_key",
        ]
    )


    train_val_overlap = (
        train_video_ids
        &
        val_video_ids
    )

    train_test_overlap = (
        train_video_ids
        &
        test_video_ids
    )

    val_test_overlap = (
        val_video_ids
        &
        test_video_ids
    )


    if train_val_overlap:

        raise AssertionError(
            "Train/Val VIDEO leakage detected. "
            f"Examples: "
            f"{sorted(train_val_overlap)[:20]}"
        )


    if train_test_overlap:

        raise AssertionError(
            "Train/Test VIDEO leakage detected. "
            f"Examples: "
            f"{sorted(train_test_overlap)[:20]}"
        )


    if val_test_overlap:

        raise AssertionError(
            "Val/Test VIDEO leakage detected. "
            f"Examples: "
            f"{sorted(val_test_overlap)[:20]}"
        )


    VIDEO_LEAKAGE_STATUS = "PASSED"

    VIDEO_LEAKAGE_REASON = (
        "All SUCCESS samples contained authoritative source_video "
        "information and Train/Val/Test video IDs were disjoint."
    )


    video_counts = (
        success_meta
        .drop_duplicates(
            [
                "video_key",
                "split",
            ]
        )
        .groupby(
            [
                "split",
                "label",
            ]
        )
        .size()
        .unstack(
            fill_value=0
        )
    )


    print(
        "[PASS] authoritative video-level split isolation"
    )


else:

    missing_video_count = (
        total_success
        -
        known_video_count
    )


    print()
    print("=" * 78)
    print("VIDEO-LEVEL LEAKAGE AUDIT")
    print("=" * 78)

    print(
        "Status : NOT AVAILABLE"
    )

    print(
        f"Known source_video rows   : "
        f"{known_video_count:,}"
    )

    print(
        f"Missing source_video rows : "
        f"{missing_video_count:,}"
    )

    print()

    print(
        "Reason:"
    )

    print(
        VIDEO_LEAKAGE_REASON
    )

    print()

    print(
        "IMPORTANT: This is NOT marked as PASSED."
    )

    print(
        "The experiment may continue because exact-content "
        "and sample-level leakage checks are still valid, "
        "but the final audit must report video-level leakage "
        "as NOT AVAILABLE unless authoritative source-video "
        "metadata is joined later."
    )


# ============================================================
# 7. SAMPLE COUNTS
# ============================================================

split_counts = (
    success_meta
    .groupby(
        [
            "split",
            "label",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)


# ============================================================
# 8. SAVE SAMPLE COUNT TABLE
# ============================================================

atomic_csv_dump(
    split_counts.reset_index(),
    METADATA_DIR
    /
    "split_sample_counts.csv",
)


# ============================================================
# 9. SAVE VIDEO COUNT TABLE WHEN AVAILABLE
# ============================================================

if video_counts is not None:

    atomic_csv_dump(
        video_counts.reset_index(),
        METADATA_DIR
        /
        "split_video_counts.csv",
    )


# ============================================================
# 10. LEAKAGE AUDIT REPORT
# ============================================================

leakage_audit = {

    "run_id": RUN_ID,

    "sample_id_split_isolation": (
        "PASSED"
    ),

    "exact_sha256_split_isolation": (
        HASH_LEAKAGE_STATUS
    ),

    "source_frame_identity_uniqueness": (
        "PASSED"
    ),

    "video_level_split_isolation": (
        VIDEO_LEAKAGE_STATUS
    ),

    "video_level_reason": (
        VIDEO_LEAKAGE_REASON
    ),

    "total_success_samples": int(
        total_success
    ),

    "samples_with_source_video": int(
        known_video_count
    ),

    "samples_without_source_video": int(
        total_success
        -
        known_video_count
    ),

    "cross_split_exact_hash_count": int(
        len(cross_split_hashes)
    ),
}


LEAKAGE_AUDIT_PATH = (
    METADATA_DIR
    /
    "leakage_audit.json"
)


atomic_json_dump(
    leakage_audit,
    LEAKAGE_AUDIT_PATH,
)


# ============================================================
# 11. FINAL AUDIT OUTPUT
# ============================================================

print()
print("=" * 78)
print("SPLIT / LEAKAGE QUALITY GATE COMPLETED")
print("=" * 78)

print()
print(
    "Sample counts:"
)

print(
    split_counts
)

print()

if video_counts is not None:

    print(
        "Unique source-video counts:"
    )

    print(
        video_counts
    )

    print()


print(
    "Sample-ID leakage        : PASSED"
)

print(
    f"Exact-content leakage    : "
    f"{HASH_LEAKAGE_STATUS}"
)

print(
    "Source-frame duplicates  : PASSED"
)

print(
    f"Video-level leakage      : "
    f"{VIDEO_LEAKAGE_STATUS}"
)

print()

print(
    "Leakage audit saved:"
)

print(
    LEAKAGE_AUDIT_PATH
)

[PASS] sample_id split isolation
[PASS] exact SHA256 content isolation
[PASS] source-frame / face identity uniqueness

VIDEO-LEVEL LEAKAGE AUDIT
Status : NOT AVAILABLE
Known source_video rows   : 0
Missing source_video rows : 2,986

Reason:
source_video is not present in the current Eye ROI filenames. Video-level leakage cannot be scientifically verified from these filenames alone.

IMPORTANT: This is NOT marked as PASSED.
The experiment may continue because exact-content and sample-level leakage checks are still valid, but the final audit must report video-level leakage as NOT AVAILABLE unless authoritative source-video metadata is joined later.

SPLIT / LEAKAGE QUALITY GATE COMPLETED

Sample counts:
label  fake  real
split            
test    156   146
train  1191  1197
val     141   155

Sample-ID leakage        : PASSED
Exact-content leakage    : PASSED
Source-frame duplicates  : PASSED
Video-level leakage      : NOT_AVAILABLE

Leakage audit saved:
/content/drive/MyDrive/AISC DeepF

In [10]:

# ============================================================
# CELL 6 — DATASET, TRAIN-ONLY AUGMENTATION, DATALOADERS
# ============================================================

IMAGE_SIZE = int(CONFIG["image_size"])
BATCH_SIZE = int(CONFIG["batch_size"])
NUM_WORKERS = int(CONFIG["num_workers"])

# ImageNet normalization because the Xception backbone is ImageNet pretrained.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

aug = CONFIG["train_augmentation"]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=float(aug["horizontal_flip_p"])),
    transforms.RandomRotation(degrees=float(aug["rotation_deg"])),
    transforms.ColorJitter(
        brightness=float(aug["brightness"]),
        contrast=float(aug["contrast"]),
    ),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ============================================================
# CELL 6 — DATASET, TRAIN-ONLY AUGMENTATION, DATALOADERS
# ============================================================

IMAGE_SIZE = int(CONFIG["image_size"])
BATCH_SIZE = int(CONFIG["batch_size"])
NUM_WORKERS = int(CONFIG["num_workers"])


# ============================================================
# NORMALIZATION
# ============================================================
#
# Xception backbone ImageNet pretrained olduğu için
# ImageNet normalization kullanıyoruz.
#
# Train / Val / Test transform izolasyonu korunur.
# Augmentation yalnızca TRAIN setine uygulanır.
# ============================================================

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406,
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225,
]


aug = CONFIG["train_augmentation"]


# ============================================================
# TRAIN TRANSFORMS
# ============================================================

train_transform = transforms.Compose(
    [
        transforms.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE,
            )
        ),

        transforms.RandomHorizontalFlip(
            p=float(
                aug["horizontal_flip_p"]
            )
        ),

        transforms.RandomRotation(
            degrees=float(
                aug["rotation_deg"]
            )
        ),

        transforms.ColorJitter(
            brightness=float(
                aug["brightness"]
            ),
            contrast=float(
                aug["contrast"]
            ),
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            IMAGENET_MEAN,
            IMAGENET_STD,
        ),
    ]
)


# ============================================================
# VALIDATION / TEST TRANSFORMS
# ============================================================
#
# No augmentation is applied to validation or test.
# ============================================================

eval_transform = transforms.Compose(
    [
        transforms.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE,
            )
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            IMAGENET_MEAN,
            IMAGENET_STD,
        ),
    ]
)


# ============================================================
# DATASET CLASS
# ============================================================

class EyeROIDataset(Dataset):
    """
    Dataset for combined Eye ROI images.

    Label convention
    ----------------
    REAL -> 0
    FAKE -> 1

    Important
    ---------
    PyTorch default DataLoader collation cannot batch None values.

    Therefore optional metadata fields such as source_video are
    converted to empty strings when unavailable.

    This conversion is ONLY for DataLoader transport.

    Empty string DOES NOT mean a valid source-video identity.
    """

    def __init__(
        self,
        frame: pd.DataFrame,
        transform,
    ):

        if frame is None:
            raise ValueError(
                "Dataset frame cannot be None."
            )

        if len(frame) == 0:
            raise ValueError(
                "Dataset frame cannot be empty."
            )

        self.frame = (
            frame
            .reset_index(drop=True)
            .copy()
        )

        self.transform = transform


        # ----------------------------------------------------
        # Required metadata columns
        # ----------------------------------------------------

        required_columns = {
            "sample_id",
            "label",
            "split",
            "output_path",
        }


        missing_columns = (
            required_columns
            .difference(
                self.frame.columns
            )
        )


        if missing_columns:

            raise ValueError(
                "EyeROIDataset metadata is missing "
                f"required columns: "
                f"{sorted(missing_columns)}"
            )


    def __len__(self):

        return len(
            self.frame
        )


    @staticmethod
    def safe_string(value):
        """
        Convert optional metadata to a DataLoader-safe string.

        None -> ""
        NaN  -> ""
        Other values -> str(value)
        """

        if value is None:

            return ""


        try:

            if pd.isna(value):

                return ""


        except Exception:

            pass


        return str(value)


    @staticmethod
    def safe_integer(
        value,
        default=-1,
    ):
        """
        Convert optional numeric metadata to int safely.
        """

        if value is None:

            return int(default)


        try:

            if pd.isna(value):

                return int(default)


        except Exception:

            pass


        try:

            return int(value)


        except Exception as exc:

            raise ValueError(
                f"Could not convert value "
                f"'{value}' to integer."
            ) from exc


    def __getitem__(
        self,
        idx,
    ):

        row = self.frame.iloc[idx]


        # ====================================================
        # IMAGE PATH
        # ====================================================

        path = Path(
            row["output_path"]
        )


        if not path.exists():

            raise FileNotFoundError(
                "Eye ROI image does not exist:\n"
                f"{path}"
            )


        # ====================================================
        # IMAGE LOAD
        # ====================================================

        try:

            with Image.open(path) as im:

                image = im.convert(
                    "RGB"
                )


        except Exception as exc:

            raise RuntimeError(
                "Could not read Eye ROI image:\n"
                f"{path}"
            ) from exc


        # ====================================================
        # TRANSFORM
        # ====================================================

        if self.transform is not None:

            image = self.transform(
                image
            )


        # ====================================================
        # LABEL
        # ====================================================

        label_name = (
            str(
                row["label"]
            )
            .lower()
            .strip()
        )


        if label_name == "real":

            label = 0.0


        elif label_name == "fake":

            label = 1.0


        else:

            raise ValueError(
                "Unexpected label detected: "
                f"'{label_name}'\n"
                f"Path: {path}"
            )


        # ====================================================
        # SAFE METADATA
        # ====================================================

        sample_id = (
            self.safe_string(
                row.get(
                    "sample_id",
                    "",
                )
            )
        )


        source_video = (
            self.safe_string(
                row.get(
                    "source_video",
                    "",
                )
            )
        )


        source_frame = (
            self.safe_string(
                row.get(
                    "source_frame",
                    "",
                )
            )
        )


        relative_path = (
            self.safe_string(
                row.get(
                    "relative_path",
                    "",
                )
            )
        )


        split_name = (
            self.safe_string(
                row.get(
                    "split",
                    "",
                )
            )
        )


        frame_index = (
            self.safe_integer(
                row.get(
                    "frame_index",
                    -1,
                )
            )
        )


        face_index = (
            self.safe_integer(
                row.get(
                    "face_index",
                    -1,
                )
            )
        )


        # ====================================================
        # RETURN SAMPLE
        # ====================================================

        sample = {

            "image": image,

            "label": torch.tensor(
                label,
                dtype=torch.float32,
            ),

            "sample_id": sample_id,

            "path": str(
                path
            ),

            # "" = unavailable
            # This is NOT treated as a video ID.
            "source_video": source_video,

            "source_frame": source_frame,

            "relative_path": relative_path,

            "split": split_name,

            "frame_index": frame_index,

            "face_index": face_index,
        }


        # ====================================================
        # NONE SAFETY CHECK
        # ====================================================

        none_keys = [
            key
            for key, value
            in sample.items()
            if value is None
        ]


        if none_keys:

            raise TypeError(
                "Dataset sample contains None values "
                f"for keys: {none_keys}\n"
                f"Path: {path}"
            )


        return sample


# ============================================================
# SPLIT DATAFRAMES
# ============================================================

train_df = (
    success_meta.loc[
        success_meta["split"]
        ==
        "train"
    ]
    .copy()
)


val_df = (
    success_meta.loc[
        success_meta["split"]
        ==
        "val"
    ]
    .copy()
)


test_df = (
    success_meta.loc[
        success_meta["split"]
        ==
        "test"
    ]
    .copy()
)


# ============================================================
# SPLIT PRESENCE QUALITY GATE
# ============================================================

for split_name, df in [

    (
        "train",
        train_df,
    ),

    (
        "val",
        val_df,
    ),

    (
        "test",
        test_df,
    ),

]:

    if len(df) == 0:

        raise AssertionError(
            f"{split_name} split is empty."
        )


    labels = set(
        df["label"]
        .astype(str)
        .str.lower()
        .unique()
    )


    if labels != {
        "real",
        "fake",
    }:

        raise AssertionError(
            f"{split_name} split must contain "
            "both REAL and FAKE classes.\n"
            f"Found: {sorted(labels)}"
        )


# ============================================================
# CREATE DATASETS
# ============================================================

train_dataset = EyeROIDataset(
    frame=train_df,
    transform=train_transform,
)


val_dataset = EyeROIDataset(
    frame=val_df,
    transform=eval_transform,
)


test_dataset = EyeROIDataset(
    frame=test_df,
    transform=eval_transform,
)


# ============================================================
# DATASET SANITY CHECK
# ============================================================

print()
print("=" * 78)
print("DATASET SANITY CHECK")
print("=" * 78)


for dataset_name, dataset in [

    (
        "TRAIN",
        train_dataset,
    ),

    (
        "VAL",
        val_dataset,
    ),

    (
        "TEST",
        test_dataset,
    ),

]:

    if len(dataset) == 0:

        raise RuntimeError(
            f"{dataset_name} dataset is empty."
        )


    sample = dataset[0]


    required_sample_keys = {

        "image",

        "label",

        "sample_id",

        "path",

        "source_video",

        "source_frame",

        "frame_index",

        "face_index",

    }


    missing_keys = (
        required_sample_keys
        -
        set(
            sample.keys()
        )
    )


    if missing_keys:

        raise AssertionError(
            f"{dataset_name} sample is missing "
            f"keys: {sorted(missing_keys)}"
        )


    if not isinstance(
        sample["image"],
        torch.Tensor,
    ):

        raise TypeError(
            f"{dataset_name}: "
            "image must be a torch.Tensor."
        )


    if not isinstance(
        sample["label"],
        torch.Tensor,
    ):

        raise TypeError(
            f"{dataset_name}: "
            "label must be a torch.Tensor."
        )


    if sample["image"].ndim != 3:

        raise AssertionError(
            f"{dataset_name}: "
            "image tensor must have shape "
            "[C, H, W]."
        )


    if (
        sample["image"].shape[0]
        !=
        3
    ):

        raise AssertionError(
            f"{dataset_name}: "
            "expected RGB tensor with 3 channels."
        )


    if not torch.isfinite(
        sample["image"]
    ).all():

        raise FloatingPointError(
            f"{dataset_name}: "
            "image tensor contains NaN/Inf."
        )


    if not torch.isfinite(
        sample["label"]
    ).all():

        raise FloatingPointError(
            f"{dataset_name}: "
            "label contains NaN/Inf."
        )


    none_keys = [

        key

        for key, value

        in sample.items()

        if value is None

    ]


    if none_keys:

        raise TypeError(
            f"{dataset_name}: "
            "sample contains None values: "
            f"{none_keys}"
        )


    print(
        f"{dataset_name:<5} | "
        f"samples={len(dataset):>6,} | "
        f"image={tuple(sample['image'].shape)} | "
        f"label={int(sample['label'].item())} | "
        f"source_video="
        f"{sample['source_video'] or '<UNAVAILABLE>'}"
    )


print("=" * 78)
print("DATASET SANITY CHECK PASSED")
print("=" * 78)


# ============================================================
# CLASS COUNTS
# ============================================================

train_fake = int(
    (
        train_df["label"]
        .astype(str)
        .str.lower()
        ==
        "fake"
    )
    .sum()
)


train_real = int(
    (
        train_df["label"]
        .astype(str)
        .str.lower()
        ==
        "real"
    )
    .sum()
)


if train_fake <= 0:

    raise RuntimeError(
        "Training split contains no FAKE samples."
    )


if train_real <= 0:

    raise RuntimeError(
        "Training split contains no REAL samples."
    )


# ============================================================
# POSITIVE CLASS WEIGHT
# ============================================================
#
# FAKE = positive class = 1
#
# pos_weight = number_of_negative / number_of_positive
#            = REAL / FAKE
# ============================================================

POS_WEIGHT = torch.tensor(
    [
        train_real
        /
        train_fake
    ],
    dtype=torch.float32,
    device=DEVICE,
)


if not torch.isfinite(
    POS_WEIGHT
).all():

    raise FloatingPointError(
        "POS_WEIGHT is NaN or Inf."
    )


# ============================================================
# REPRODUCIBLE DATALOADER WORKER SEED
# ============================================================

def seed_worker(
    worker_id,
):

    worker_seed = (
        torch.initial_seed()
        %
        2**32
    )

    np.random.seed(
        worker_seed
    )

    random.seed(
        worker_seed
    )


loader_generator = (
    torch.Generator()
)


loader_generator.manual_seed(
    SEED
)


# ============================================================
# DATALOADER COMMON CONFIG
# ============================================================

loader_kwargs = {

    "batch_size": BATCH_SIZE,

    "num_workers": NUM_WORKERS,

    "pin_memory": (
        DEVICE.type
        ==
        "cuda"
    ),

    "persistent_workers": (
        NUM_WORKERS
        >
        0
    ),

    "worker_init_fn": seed_worker,

}


# ============================================================
# TRAIN DATALOADER
# ============================================================

train_loader = DataLoader(

    train_dataset,

    shuffle=True,

    generator=loader_generator,

    drop_last=False,

    **loader_kwargs,
)


# ============================================================
# VALIDATION DATALOADER
# ============================================================

val_loader = DataLoader(

    val_dataset,

    shuffle=False,

    drop_last=False,

    **loader_kwargs,
)


# ============================================================
# TEST DATALOADER
# ============================================================

test_loader = DataLoader(

    test_dataset,

    shuffle=False,

    drop_last=False,

    **loader_kwargs,
)


# ============================================================
# DATALOADER COLLATION QUALITY GATE
# ============================================================
#
# This catches the exact NoneType error you encountered BEFORE
# the model smoke test begins.
# ============================================================

print()
print("=" * 78)
print("DATALOADER COLLATION CHECK")
print("=" * 78)


try:

    test_batch = next(
        iter(
            train_loader
        )
    )


except Exception as exc:

    raise RuntimeError(
        "Train DataLoader could not create "
        "the first batch."
    ) from exc


required_batch_keys = {

    "image",

    "label",

    "sample_id",

    "path",

    "source_video",

    "source_frame",

    "frame_index",

    "face_index",

}


missing_batch_keys = (
    required_batch_keys
    -
    set(
        test_batch.keys()
    )
)


if missing_batch_keys:

    raise AssertionError(
        "DataLoader batch is missing keys: "
        f"{sorted(missing_batch_keys)}"
    )


if not isinstance(
    test_batch["image"],
    torch.Tensor,
):

    raise TypeError(
        "DataLoader image batch must "
        "be a torch.Tensor."
    )


if not isinstance(
    test_batch["label"],
    torch.Tensor,
):

    raise TypeError(
        "DataLoader label batch must "
        "be a torch.Tensor."
    )


if (
    test_batch["image"].ndim
    !=
    4
):

    raise AssertionError(
        "Expected DataLoader image batch "
        "shape [B, C, H, W]."
    )


if (
    test_batch["image"].shape[1]
    !=
    3
):

    raise AssertionError(
        "Expected RGB batch with 3 channels."
    )


if not torch.isfinite(
    test_batch["image"]
).all():

    raise FloatingPointError(
        "First training batch contains "
        "NaN/Inf image values."
    )


if not torch.isfinite(
    test_batch["label"]
).all():

    raise FloatingPointError(
        "First training batch contains "
        "NaN/Inf labels."
    )


print(
    "First batch image shape:",
    tuple(
        test_batch["image"].shape
    ),
)


print(
    "First batch label shape:",
    tuple(
        test_batch["label"].shape
    ),
)


print(
    "First batch source_video example:",
    (
        test_batch["source_video"][0]
        or
        "<UNAVAILABLE>"
    ),
)


print(
    "DATALOADER COLLATION CHECK PASSED"
)


# Free the validation batch before training.
del test_batch

gc.collect()


# ============================================================
# FINAL DATA SUMMARY
# ============================================================

print()
print("=" * 78)
print("EYE ROI DATA PIPELINE READY")
print("=" * 78)


print(
    f"Image size        : "
    f"{IMAGE_SIZE} x {IMAGE_SIZE}"
)


print(
    f"Batch size        : "
    f"{BATCH_SIZE}"
)


print(
    f"Num workers       : "
    f"{NUM_WORKERS}"
)


print(
    f"Train samples     : "
    f"{len(train_dataset):,}"
)


print(
    f"Validation samples: "
    f"{len(val_dataset):,}"
)


print(
    f"Test samples      : "
    f"{len(test_dataset):,}"
)


print(
    f"Train REAL        : "
    f"{train_real:,}"
)


print(
    f"Train FAKE        : "
    f"{train_fake:,}"
)


print(
    f"BCE pos_weight    : "
    f"{float(POS_WEIGHT.item()):.6f}"
)


print(
    f"Device            : "
    f"{DEVICE}"
)


print(
    f"Pin memory        : "
    f"{DEVICE.type == 'cuda'}"
)


print(
    "=" * 78
)


DATASET SANITY CHECK
TRAIN | samples= 2,388 | image=(3, 299, 299) | label=0 | source_video=<UNAVAILABLE>
VAL   | samples=   296 | image=(3, 299, 299) | label=0 | source_video=<UNAVAILABLE>
TEST  | samples=   302 | image=(3, 299, 299) | label=0 | source_video=<UNAVAILABLE>
DATASET SANITY CHECK PASSED

DATALOADER COLLATION CHECK
First batch image shape: (16, 3, 299, 299)
First batch label shape: (16,)
First batch source_video example: <UNAVAILABLE>
DATALOADER COLLATION CHECK PASSED

EYE ROI DATA PIPELINE READY
Image size        : 299 x 299
Batch size        : 16
Num workers       : 2
Train samples     : 2,388
Validation samples: 296
Test samples      : 302
Train REAL        : 1,197
Train FAKE        : 1,191
BCE pos_weight    : 1.005038
Device            : cuda
Pin memory        : True


In [11]:

# ============================================================
# CELL 7 — MODEL BUILD + SMOKE TEST QUALITY GATE
# ============================================================

MODEL_NAME = CONFIG["model_name"]

model = timm.create_model(
    MODEL_NAME,
    pretrained=bool(CONFIG["pretrained"]),
    num_classes=1,
)
model = model.to(DEVICE)

# Stage 1: freeze all backbone parameters, train only classifier.
for p in model.parameters():
    p.requires_grad = False

classifier = model.get_classifier()
if classifier is None:
    raise RuntimeError("Could not locate Xception classifier head via timm.get_classifier().")

for p in classifier.parameters():
    p.requires_grad = True

criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)

def build_optimizer(lr: float):
    trainable = [p for p in model.parameters() if p.requires_grad]
    if not trainable:
        raise RuntimeError("No trainable parameters.")
    return torch.optim.AdamW(
        trainable,
        lr=float(lr),
        weight_decay=float(CONFIG["weight_decay"]),
    )

def build_scheduler(optimizer):
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=float(CONFIG["scheduler_factor"]),
        patience=int(CONFIG["scheduler_patience"]),
    )

# Smoke test: exactly two batches forward + backward before full training.
smoke_optimizer = build_optimizer(CONFIG["initial_lr"])
model.train()
smoke_losses = []

for batch_idx, batch in enumerate(train_loader):
    if batch_idx >= 2:
        break

    images = batch["image"].to(DEVICE, non_blocking=True)
    targets = batch["label"].to(DEVICE, non_blocking=True).view(-1, 1)

    smoke_optimizer.zero_grad(set_to_none=True)
    logits = model(images)
    loss = criterion(logits, targets)

    if not torch.isfinite(loss):
        raise FloatingPointError(f"Smoke test produced non-finite loss: {loss.item()}")

    loss.backward()

    for name, param in model.named_parameters():
        if param.grad is not None and not torch.isfinite(param.grad).all():
            raise FloatingPointError(f"Non-finite gradient in smoke test: {name}")

    smoke_optimizer.step()
    smoke_losses.append(float(loss.item()))

if len(smoke_losses) != 2:
    raise RuntimeError("Smoke test could not read two batches.")

# Rebuild a clean model after smoke test so smoke optimization never contaminates training.
del model, smoke_optimizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = timm.create_model(
    MODEL_NAME,
    pretrained=bool(CONFIG["pretrained"]),
    num_classes=1,
).to(DEVICE)

for p in model.parameters():
    p.requires_grad = False
classifier = model.get_classifier()
for p in classifier.parameters():
    p.requires_grad = True

print("Smoke test PASSED. Losses:", smoke_losses)
print("Trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Total parameters:",
      sum(p.numel() for p in model.parameters()))


Smoke test PASSED. Losses: [0.6864856481552124, 0.7040135860443115]
Trainable parameters: 2049
Total parameters: 20809001


In [12]:

# ============================================================
# CELL 8 — TRAINING / EVALUATION ENGINE WITH ATOMIC RESUME
# ============================================================

def binary_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_prob = np.asarray(y_prob, dtype=np.float64)
    y_pred = (y_prob >= threshold).astype(np.int64)

    result = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }

    if len(np.unique(y_true)) == 2:
        result["roc_auc"] = float(roc_auc_score(y_true, y_prob))
        result["pr_auc"] = float(average_precision_score(y_true, y_prob))
    else:
        result["roc_auc"] = float("nan")
        result["pr_auc"] = float("nan")

    return result

def run_epoch(loader, optimizer=None, scaler=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    all_y = []
    all_prob = []

    context = torch.enable_grad if is_train else torch.no_grad

    with context():
        for batch in tqdm(loader, leave=False):
            images = batch["image"].to(DEVICE, non_blocking=True)
            targets = batch["label"].to(DEVICE, non_blocking=True).view(-1, 1)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16 if DEVICE.type == "cuda" else torch.bfloat16,
                enabled=AMP_ENABLED,
            ):
                logits = model(images)
                loss = criterion(logits, targets)

            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss detected: {loss.item()}")

            if is_train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)

                grad_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=float(CONFIG["gradient_clip_norm"]),
                )

                if not torch.isfinite(torch.as_tensor(grad_norm)):
                    raise FloatingPointError("Non-finite gradient norm detected.")

                scaler.step(optimizer)
                scaler.update()

            probs = torch.sigmoid(logits.detach()).view(-1).cpu().numpy()
            y = targets.detach().view(-1).cpu().numpy()

            total_loss += float(loss.item()) * len(y)
            all_y.extend(y.tolist())
            all_prob.extend(probs.tolist())

    epoch_loss = total_loss / len(loader.dataset)
    metrics = binary_metrics(all_y, all_prob, threshold=0.5)
    return epoch_loss, metrics, np.asarray(all_y), np.asarray(all_prob)

def train_stage(stage_name, epochs, lr, stage_dir, resume=True):
    stage_dir.mkdir(parents=True, exist_ok=True)

    optimizer = build_optimizer(lr)
    scheduler = build_scheduler(optimizer)
    scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

    last_path = stage_dir / "last.ckpt"
    best_path = stage_dir / "best.ckpt"

    start_epoch = 1
    best_score = -np.inf
    wait = 0
    history = []

    history_path = LOG_DIR / f"{stage_name}_history.csv"
    if history_path.exists():
        history = pd.read_csv(history_path).to_dict("records")

    if resume and last_path.exists():
        ckpt = torch.load(last_path, map_location=DEVICE, weights_only=False)
        if ckpt["stage"] != stage_name:
            raise RuntimeError(f"Checkpoint stage mismatch: {ckpt['stage']} vs {stage_name}")

        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        scaler.load_state_dict(ckpt["scaler_state_dict"])
        restore_rng_state(ckpt["rng_state"])

        start_epoch = int(ckpt["epoch"]) + 1
        best_score = float(ckpt["best_metric_score"])
        wait = int(ckpt.get("early_stopping_wait", 0))

        print(f"Resuming {stage_name} from epoch {start_epoch}")

    for epoch in range(start_epoch, int(epochs) + 1):
        train_loss, train_m, _, _ = run_epoch(
            train_loader,
            optimizer=optimizer,
            scaler=scaler,
        )
        val_loss, val_m, _, _ = run_epoch(val_loader)

        score = val_m["roc_auc"]
        if not np.isfinite(score):
            score = val_m["f1"]

        scheduler.step(score)

        row = {
            "stage": stage_name,
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            "train_loss": train_loss,
            "val_loss": val_loss,
            **{f"train_{k}": v for k, v in train_m.items()},
            **{f"val_{k}": v for k, v in val_m.items()},
        }
        history.append(row)
        atomic_csv_dump(pd.DataFrame(history), history_path)

        improved = score > best_score + 1e-6
        if improved:
            best_score = score
            wait = 0
        else:
            wait += 1

        state = {
            "epoch": epoch,
            "stage": stage_name,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_metric_score": best_score,
            "early_stopping_wait": wait,
            "config": CONFIG,
            "rng_state": capture_rng_state(),
        }

        atomic_save_checkpoint(state, last_path)
        atomic_save_checkpoint(state, stage_dir / f"epoch_{epoch}.ckpt")
        rotate_epoch_checkpoints(
            stage_dir,
            keep_last_n=int(CONFIG["checkpoint_keep_last_n"]),
        )

        if improved:
            atomic_save_checkpoint(state, best_path)

        print(
            f"[{stage_name}] epoch={epoch:02d} "
            f"train_loss={train_loss:.5f} val_loss={val_loss:.5f} "
            f"val_auc={val_m['roc_auc']:.5f} val_f1={val_m['f1']:.5f}"
        )

        if wait >= int(CONFIG["early_stopping_patience"]):
            print(f"Early stopping: {stage_name}")
            break

    if not best_path.exists():
        raise RuntimeError(f"Best checkpoint was not created for {stage_name}")

    best_ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(best_ckpt["model_state_dict"])

    return {
        "stage": stage_name,
        "best_metric_score": float(best_ckpt["best_metric_score"]),
        "best_checkpoint": str(best_path),
        "history_path": str(history_path),
    }

print("Training engine ready.")


Training engine ready.


In [13]:

# ============================================================
# CELL 9 — STAGE 1: FROZEN XCEPTION
# ============================================================

for p in model.parameters():
    p.requires_grad = False
classifier = model.get_classifier()
for p in classifier.parameters():
    p.requires_grad = True

FROZEN_DIR = CHECKPOINT_DIR / "frozen"

frozen_summary = train_stage(
    stage_name="frozen",
    epochs=int(CONFIG["initial_epochs"]),
    lr=float(CONFIG["initial_lr"]),
    stage_dir=FROZEN_DIR,
    resume=True,
)

atomic_json_dump(
    frozen_summary,
    METRICS_DIR / "frozen_training_summary.json",
)

print(frozen_summary)


/tmp/ipykernel_3035/3736086830.py:87: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)


  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/19 [00:00<?, ?it/s]

[frozen] epoch=01 train_loss=0.69097 val_loss=0.69394 val_auc=0.50176 val_f1=0.40664


  0%|          | 0/150 [00:00<?, ?it/s]

...

  0%|          | 0/19 [00:00<?, ?it/s]

[frozen] epoch=11 train_loss=0.65141 val_loss=0.68245 val_auc=0.59693 val_f1=0.59941


  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/19 [00:00<?, ?it/s]

[frozen] epoch=12 train_loss=0.65591 val_loss=0.68107 val_auc=0.60355 val_f1=0.60843
{'stage': 'frozen', 'best_metric_score': 0.6035460992907802, 'best_checkpoint': '/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/20260807_1004_eye_xception_seed42/checkpoints/frozen/best.ckpt', 'history_path': '/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/20260807_1004_eye_xception_seed42/logs/frozen_history.csv'}


In [15]:
# ============================================================
# CELL 10 — STAGE 2: STABLE FULL XCEPTION FINE-TUNING
# ============================================================

print("=" * 78)
print("STAGE 2 — FULL XCEPTION FINE-TUNING")
print("=" * 78)


# ============================================================
# 1. LOAD BEST FROZEN CHECKPOINT
# ============================================================

FROZEN_BEST_PATH = (
    FROZEN_DIR
    /
    "best.ckpt"
)


if not FROZEN_BEST_PATH.exists():

    raise FileNotFoundError(
        "Frozen best checkpoint does not exist:\n"
        f"{FROZEN_BEST_PATH}"
    )


frozen_best = torch.load(
    FROZEN_BEST_PATH,
    map_location=DEVICE,
    weights_only=False,
)


required_checkpoint_keys = {
    "model_state_dict",
    "epoch",
    "stage",
    "best_metric_score",
}


missing_checkpoint_keys = (
    required_checkpoint_keys
    -
    set(frozen_best.keys())
)


if missing_checkpoint_keys:

    raise RuntimeError(
        "Frozen checkpoint is missing required keys: "
        f"{sorted(missing_checkpoint_keys)}"
    )


model.load_state_dict(
    frozen_best["model_state_dict"]
)


print(
    "Loaded frozen checkpoint:"
)

print(
    FROZEN_BEST_PATH
)

print(
    f"Frozen best score: "
    f"{float(frozen_best['best_metric_score']):.6f}"
)


# ============================================================
# 2. UNFREEZE FULL XCEPTION
# ============================================================

for parameter in model.parameters():

    parameter.requires_grad = True


trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


total_parameters = sum(
    p.numel()
    for p in model.parameters()
)


if trainable_parameters == 0:

    raise RuntimeError(
        "No trainable parameters after unfreezing Xception."
    )


print()
print(
    f"Trainable parameters: "
    f"{trainable_parameters:,}"
)

print(
    f"Total parameters    : "
    f"{total_parameters:,}"
)


# ============================================================
# 3. STABILIZE BATCH NORMALIZATION
# ============================================================
#
# During full fine-tuning, Xception BatchNorm statistics can become
# unstable with relatively small batches.
#
# We keep BatchNorm running statistics frozen while allowing
# affine parameters to remain trainable.
# ============================================================

def freeze_batchnorm_running_stats(module):

    if isinstance(
        module,
        nn.modules.batchnorm._BatchNorm,
    ):

        module.eval()


batchnorm_count = 0


for module in model.modules():

    if isinstance(
        module,
        nn.modules.batchnorm._BatchNorm,
    ):

        batchnorm_count += 1

        module.eval()


print(
    f"BatchNorm layers with frozen running stats: "
    f"{batchnorm_count}"
)


# ============================================================
# 4. DISABLE AMP FOR FULL FINE-TUNING
# ============================================================
#
# Frozen training succeeded with AMP.
#
# However, the first full-finetuning backward pass generated
# non-finite gradients.
#
# Therefore Stage 2 uses FP32 deliberately.
#
# We do NOT suppress the numerical quality gate.
# ============================================================

PREVIOUS_AMP_ENABLED = AMP_ENABLED

AMP_ENABLED = False


print()
print(
    "Automatic Mixed Precision:"
)

print(
    "DISABLED for Stage 2 (FP32 fine-tuning)"
)


# ============================================================
# 5. USE A MORE CONSERVATIVE FINE-TUNING LR
# ============================================================
#
# Original configuration:
#     finetune_lr = 1e-5
#
# Full-backbone fine-tuning is more sensitive than head-only
# training, therefore use a conservative LR.
# ============================================================

ORIGINAL_FINETUNE_LR = float(
    CONFIG["finetune_lr"]
)


STABLE_FINETUNE_LR = min(
    ORIGINAL_FINETUNE_LR,
    5e-6,
)


print(
    f"Configured fine-tune LR : "
    f"{ORIGINAL_FINETUNE_LR:.2e}"
)

print(
    f"Effective fine-tune LR  : "
    f"{STABLE_FINETUNE_LR:.2e}"
)


# ============================================================
# 6. REMOVE PARTIAL FAILED FINETUNE CHECKPOINT IF NECESSARY
# ============================================================
#
# Your failure occurred before epoch 1 completed, therefore no
# valid epoch checkpoint should normally exist.
#
# We inspect instead of silently overwriting valid checkpoints.
# ============================================================

FINETUNE_DIR = (
    CHECKPOINT_DIR
    /
    "finetune"
)


FINETUNE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


finetune_last_path = (
    FINETUNE_DIR
    /
    "last.ckpt"
)


if finetune_last_path.exists():

    existing_ckpt = torch.load(
        finetune_last_path,
        map_location="cpu",
        weights_only=False,
    )


    completed_epoch = int(
        existing_ckpt.get(
            "epoch",
            0,
        )
    )


    print()
    print(
        "Existing fine-tuning checkpoint detected:"
    )

    print(
        finetune_last_path
    )

    print(
        f"Completed epoch: "
        f"{completed_epoch}"
    )

    print(
        "Training will resume from this valid checkpoint."
    )


else:

    print()
    print(
        "No completed fine-tuning checkpoint found."
    )

    print(
        "Stage 2 will start from frozen best.ckpt."
    )


# ============================================================
# 7. PRE-FINETUNE FP32 FORWARD/BACKWARD QUALITY GATE
# ============================================================
#
# Before starting an entire epoch, test one batch using the exact
# Stage-2 numerical mode.
#
# This is separate from the original smoke test.
# ============================================================

print()
print("=" * 78)
print("STAGE 2 NUMERICAL SMOKE TEST")
print("=" * 78)


probe_optimizer = torch.optim.AdamW(
    [
        p
        for p in model.parameters()
        if p.requires_grad
    ],
    lr=STABLE_FINETUNE_LR,
    weight_decay=float(
        CONFIG["weight_decay"]
    ),
)


probe_batch = next(
    iter(train_loader)
)


probe_images = (
    probe_batch["image"]
    .to(
        DEVICE,
        non_blocking=True,
    )
)


probe_targets = (
    probe_batch["label"]
    .to(
        DEVICE,
        non_blocking=True,
    )
    .view(-1, 1)
)


probe_optimizer.zero_grad(
    set_to_none=True
)


# Full FP32 forward pass.
probe_logits = model(
    probe_images
)


if not torch.isfinite(
    probe_logits
).all():

    raise FloatingPointError(
        "Stage-2 FP32 smoke test produced "
        "non-finite logits."
    )


probe_loss = criterion(
    probe_logits,
    probe_targets,
)


if not torch.isfinite(
    probe_loss
):

    raise FloatingPointError(
        "Stage-2 FP32 smoke test produced "
        f"non-finite loss: {probe_loss.item()}"
    )


probe_loss.backward()


nonfinite_gradient_names = []


for name, parameter in model.named_parameters():

    if parameter.grad is None:

        continue


    if not torch.isfinite(
        parameter.grad
    ).all():

        nonfinite_gradient_names.append(
            name
        )


if nonfinite_gradient_names:

    raise FloatingPointError(
        "Stage-2 FP32 smoke test still produced "
        "non-finite gradients.\n"
        "Affected parameters:\n- "
        +
        "\n- ".join(
            nonfinite_gradient_names[:30]
        )
    )


probe_grad_norm = (
    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=float(
            CONFIG["gradient_clip_norm"]
        ),
    )
)


if not torch.isfinite(
    torch.as_tensor(
        probe_grad_norm
    )
):

    raise FloatingPointError(
        "Stage-2 FP32 smoke-test gradient norm "
        "is non-finite."
    )


print(
    f"Probe loss          : "
    f"{float(probe_loss.item()):.6f}"
)

print(
    f"Probe gradient norm : "
    f"{float(probe_grad_norm):.6f}"
)

print(
    "STAGE 2 NUMERICAL SMOKE TEST PASSED"
)


# ============================================================
# 8. IMPORTANT — RESET MODEL AFTER PROBE
# ============================================================
#
# The probe performed backward() but NOT optimizer.step().
# Nevertheless gradients are cleared and the frozen-best model
# is reloaded so the real training begins from an absolutely
# clean state.
# ============================================================

del probe_optimizer
del probe_batch
del probe_images
del probe_targets
del probe_logits
del probe_loss


model.zero_grad(
    set_to_none=True
)


model.load_state_dict(
    frozen_best["model_state_dict"]
)


for parameter in model.parameters():

    parameter.requires_grad = True


# BatchNorm running statistics remain frozen.
for module in model.modules():

    if isinstance(
        module,
        nn.modules.batchnorm._BatchNorm,
    ):

        module.eval()


gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ============================================================
# 9. RUN STABLE FULL FINE-TUNING
# ============================================================

finetune_summary = train_stage(

    stage_name="finetune",

    epochs=int(
        CONFIG["finetune_epochs"]
    ),

    lr=STABLE_FINETUNE_LR,

    stage_dir=FINETUNE_DIR,

    resume=True,
)


# ============================================================
# 10. SAVE TRAINING SUMMARY
# ============================================================

finetune_summary[
    "amp_enabled"
] = False


finetune_summary[
    "configured_finetune_lr"
] = ORIGINAL_FINETUNE_LR


finetune_summary[
    "effective_finetune_lr"
] = STABLE_FINETUNE_LR


finetune_summary[
    "batchnorm_running_stats"
] = "frozen"


atomic_json_dump(

    finetune_summary,

    METRICS_DIR
    /
    "finetune_training_summary.json",
)


# ============================================================
# 11. RESTORE GLOBAL AMP SETTING
# ============================================================

AMP_ENABLED = (
    PREVIOUS_AMP_ENABLED
)


# ============================================================
# FINAL OUTPUT
# ============================================================

print()
print("=" * 78)
print("FULL XCEPTION FINE-TUNING COMPLETED")
print("=" * 78)

print(
    finetune_summary
)

STAGE 2 — FULL XCEPTION FINE-TUNING
Loaded frozen checkpoint:
/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/20260807_1004_eye_xception_seed42/checkpoints/frozen/best.ckpt
Frozen best score: 0.603546

Trainable parameters: 20,809,001
Total parameters    : 20,809,001
BatchNorm layers with frozen running stats: 40

Automatic Mixed Precision:
DISABLED for Stage 2 (FP32 fine-tuning)
Configured fine-tune LR : 1.00e-05
Effective fine-tune LR  : 5.00e-06

No completed fine-tuning checkpoint found.
Stage 2 will start from frozen best.ckpt.

STAGE 2 NUMERICAL SMOKE TEST
Probe loss          : 0.663541
Probe gradient norm : 6.744028
STAGE 2 NUMERICAL SMOKE TEST PASSED


/tmp/ipykernel_3035/3736086830.py:87: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)


  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/19 [00:00<?, ?it/s]

[finetune] epoch=01 train_loss=0.64783 val_loss=0.66626 val_auc=0.64356 val_f1=0.63192


...

  0%|          | 0/19 [00:00<?, ?it/s]

[finetune] epoch=07 train_loss=0.58542 val_loss=0.62134 val_auc=0.72116 val_f1=0.65505


  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/19 [00:00<?, ?it/s]

[finetune] epoch=08 train_loss=0.58076 val_loss=0.61956 val_auc=0.72523 val_f1=0.67320

FULL XCEPTION FINE-TUNING COMPLETED
{'stage': 'finetune', 'best_metric_score': 0.7252345001143902, 'best_checkpoint': '/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/20260807_1004_eye_xception_seed42/checkpoints/finetune/best.ckpt', 'history_path': '/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/20260807_1004_eye_xception_seed42/logs/finetune_history.csv', 'amp_enabled': False, 'configured_finetune_lr': 1e-05, 'effective_finetune_lr': 5e-06, 'batchnorm_running_stats': 'frozen'}


In [16]:

# ============================================================
# CELL 11 — VALIDATION-ONLY MODEL + THRESHOLD SELECTION
# ============================================================

candidates = {
    "frozen": FROZEN_DIR / "best.ckpt",
    "finetune": FINETUNE_DIR / "best.ckpt",
}

validation_results = {}

for stage, ckpt_path in candidates.items():
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])

    val_loss, val_m, val_y, val_prob = run_epoch(val_loader)
    validation_results[stage] = {
        "checkpoint": str(ckpt_path),
        "val_loss": float(val_loss),
        **val_m,
    }

selected_stage = max(
    validation_results,
    key=lambda s: (
        validation_results[s]["roc_auc"]
        if np.isfinite(validation_results[s]["roc_auc"])
        else validation_results[s]["f1"]
    ),
)
SELECTED_CHECKPOINT = Path(validation_results[selected_stage]["checkpoint"])

# Load selected model and compute validation predictions again.
selected_ckpt = torch.load(
    SELECTED_CHECKPOINT,
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(selected_ckpt["model_state_dict"])
_, _, val_y, val_prob = run_epoch(val_loader)

# Threshold is selected ONLY on validation set.
grid_points = int(CONFIG["threshold_grid_points"])
thresholds = np.linspace(0.05, 0.95, grid_points)

threshold_table = []
for threshold in thresholds:
    m = binary_metrics(val_y, val_prob, threshold=float(threshold))
    threshold_table.append({
        "threshold": float(threshold),
        **m,
    })

threshold_df = pd.DataFrame(threshold_table)
best_idx = threshold_df["f1"].idxmax()
SELECTED_THRESHOLD = float(threshold_df.loc[best_idx, "threshold"])

atomic_csv_dump(
    threshold_df,
    METRICS_DIR / "validation_threshold_search.csv",
)

selection_summary = {
    "selection_metric": "validation_roc_auc",
    "selected_stage": selected_stage,
    "selected_checkpoint": str(SELECTED_CHECKPOINT),
    "selected_threshold": SELECTED_THRESHOLD,
    "validation_results": validation_results,
    "test_used_for_selection": False,
}
atomic_json_dump(
    selection_summary,
    METRICS_DIR / "model_selection.json",
)

# Copy the selected best checkpoint to the run-level canonical best.ckpt.
shutil.copy2(SELECTED_CHECKPOINT, CHECKPOINT_DIR / "best.ckpt")

print(json.dumps(selection_summary, indent=2, ensure_ascii=False))


  0%|          | 0/19 [00:00<?, ?it/s]

  0%|          | 0/19 [00:00<?, ?it/s]

  0%|          | 0/19 [00:00<?, ?it/s]

{
  "selection_metric": "validation_roc_auc",
  "selected_stage": "finetune",
  "selected_checkpoint": "/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/20260807_1004_eye_xception_seed42/checkpoints/finetune/best.ckpt",
  "selected_threshold": 0.30499999999999994,
  "validation_results": {
    "frozen": {
      "checkpoint": "/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/20260807_1004_eye_xception_seed42/checkpoints/frozen/best.ckpt",
      "val_loss": 0.6810716165078653,
      "accuracy": 0.5608108108108109,
      "balanced_accuracy": 0.5678334477236331,
      "precision": 0.5287958115183246,
      "recall": 0.7163120567375887,
      "f1": 0.608433734939759,
      "roc_auc": 0.6035460992907802,
      "pr_auc": 0.5794403534456058
    },
    "finetune": {
      "checkpoint": "/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/20260807_1004_eye_xception_seed42/checkpoints/finetune/be

In [17]:

# ============================================================
# CELL 12 — FINAL TEST INFERENCE + METRICS + INFERENCE TEST
# ============================================================

# Inference test: reload best.ckpt from scratch into a fresh model.
final_model = timm.create_model(
    MODEL_NAME,
    pretrained=False,
    num_classes=1,
).to(DEVICE)

final_ckpt = torch.load(
    CHECKPOINT_DIR / "best.ckpt",
    map_location=DEVICE,
    weights_only=False,
)
final_model.load_state_dict(final_ckpt["model_state_dict"])
final_model.eval()

# Temporarily switch global model reference for shared evaluator.
training_model_reference = model
model = final_model

test_loss, _, y_true, test_probabilities = run_epoch(test_loader)
y_pred = (test_probabilities >= SELECTED_THRESHOLD).astype(np.int64)

test_metrics = binary_metrics(
    y_true,
    test_probabilities,
    threshold=SELECTED_THRESHOLD,
)
test_metrics["test_loss"] = float(test_loss)
test_metrics["threshold"] = float(SELECTED_THRESHOLD)
test_metrics["sample_count"] = int(len(y_true))
test_metrics["selected_stage"] = selected_stage

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

test_metrics.update({
    "true_negative": int(tn),
    "false_positive": int(fp),
    "false_negative": int(fn),
    "true_positive": int(tp),
    "specificity": float(tn / (tn + fp)) if (tn + fp) else 0.0,
})

predictions = test_df.reset_index(drop=True)[
    ["sample_id", "source_video", "label", "output_path"]
].copy()
predictions["y_true"] = y_true.astype(int)
predictions["p_fake"] = test_probabilities.astype(float)
predictions["y_pred"] = y_pred.astype(int)
predictions["threshold"] = SELECTED_THRESHOLD
predictions["correct"] = predictions["y_true"] == predictions["y_pred"]

atomic_csv_dump(
    predictions,
    PREDICTIONS_DIR / "test_predictions.csv",
)
atomic_json_dump(
    test_metrics,
    METRICS_DIR / "final_test_metrics.json",
)

report = classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["REAL", "FAKE"],
    digits=5,
    zero_division=0,
)
report_tmp = METRICS_DIR / "classification_report.txt.tmp"
report_tmp.write_text(report, encoding="utf-8")
os.replace(report_tmp, METRICS_DIR / "classification_report.txt")

# Restore original reference, though no more training will occur.
model = training_model_reference

print("=" * 78)
print("FINAL TEST METRICS")
print("=" * 78)
for k, v in test_metrics.items():
    print(f"{k:<24}: {v}")
print("\n", report)


  0%|          | 0/19 [00:00<?, ?it/s]

FINAL TEST METRICS
accuracy                : 0.5728476821192053
balanced_accuracy       : 0.5588777660695469
precision               : 0.5483870967741935
recall                  : 0.9807692307692307
f1                      : 0.7034482758620689
roc_auc                 : 0.6769845451352301
pr_auc                  : 0.6865449635830163
test_loss               : 0.6510057757232363
threshold               : 0.30499999999999994
sample_count            : 302
selected_stage          : finetune
true_negative           : 20
false_positive          : 126
false_negative          : 3
true_positive           : 153
specificity             : 0.136986301369863

               precision    recall  f1-score   support

        REAL    0.86957   0.13699   0.23669       146
        FAKE    0.54839   0.98077   0.70345       156

    accuracy                        0.57285       302
   macro avg    0.70898   0.55888   0.47007       302
weighted avg    0.70366   0.57285   0.47780       302



In [18]:

# ============================================================
# CELL 13 — REPORT FIGURES (ENGLISH, >=600 px SHORT SIDE)
# ============================================================

def save_figure(fig, filename: str):
    path = FIGURES_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    with Image.open(path) as im:
        if min(im.size) < 600:
            raise AssertionError(
                f"Figure resolution below SSOT minimum: {path} -> {im.size}"
            )
    return path

# 1) Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 7), dpi=150)
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
im = ax.imshow(cm)
ax.set_title("Xception Eye ROI — Confusion Matrix", fontsize=14)
ax.set_xlabel("Predicted Class", fontsize=11)
ax.set_ylabel("True Class", fontsize=11)
ax.set_xticks([0, 1], labels=["REAL", "FAKE"])
ax.set_yticks([0, 1], labels=["REAL", "FAKE"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=13)
fig.colorbar(im, ax=ax)
save_figure(fig, "01_confusion_matrix.png")

# 2) ROC Curve
fpr, tpr, _ = roc_curve(y_true, test_probabilities)
fig, ax = plt.subplots(figsize=(10, 7), dpi=150)
ax.plot(fpr, tpr, linewidth=2, label=f"Xception Eye ROI (AUC = {test_metrics['roc_auc']:.4f})")
ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.5, label="Random Classifier")
ax.set_title("ROC Curve — Xception Eye ROI", fontsize=14)
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.25)
save_figure(fig, "02_roc_curve.png")

# 3) Precision–Recall Curve
pr_precision, pr_recall, _ = precision_recall_curve(y_true, test_probabilities)
fig, ax = plt.subplots(figsize=(10, 7), dpi=150)
ax.plot(
    pr_recall,
    pr_precision,
    linewidth=2,
    label=f"Xception Eye ROI (AP = {test_metrics['pr_auc']:.4f})",
)
ax.set_title("Precision–Recall Curve — Xception Eye ROI", fontsize=14)
ax.set_xlabel("Recall", fontsize=11)
ax.set_ylabel("Precision", fontsize=11)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.25)
save_figure(fig, "03_precision_recall_curve.png")

# 4–5) Training curves
history_frames = []
for history_file in [
    LOG_DIR / "frozen_history.csv",
    LOG_DIR / "finetune_history.csv",
]:
    if history_file.exists():
        history_frames.append(pd.read_csv(history_file))

history_all = pd.concat(history_frames, ignore_index=True)

fig, ax = plt.subplots(figsize=(10, 7), dpi=150)
x = np.arange(1, len(history_all) + 1)
ax.plot(x, history_all["train_loss"], label="Training Loss", linewidth=2)
ax.plot(x, history_all["val_loss"], label="Validation Loss", linewidth=2, linestyle="--")
ax.set_title("Training and Validation Loss", fontsize=14)
ax.set_xlabel("Epoch", fontsize=11)
ax.set_ylabel("Loss", fontsize=11)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.25)
save_figure(fig, "04_loss_curve.png")

fig, ax = plt.subplots(figsize=(10, 7), dpi=150)
ax.plot(x, history_all["train_roc_auc"], label="Training ROC-AUC", linewidth=2)
ax.plot(x, history_all["val_roc_auc"], label="Validation ROC-AUC", linewidth=2, linestyle="--")
ax.set_title("Training and Validation ROC-AUC", fontsize=14)
ax.set_xlabel("Epoch", fontsize=11)
ax.set_ylabel("ROC-AUC", fontsize=11)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.25)
save_figure(fig, "05_roc_auc_curve.png")

# 6) Final metrics
metric_names = ["accuracy", "precision", "recall", "specificity", "f1", "roc_auc", "pr_auc"]
metric_values = [test_metrics[m] for m in metric_names]
display_names = ["Accuracy", "Precision", "Recall", "Specificity", "F1", "ROC-AUC", "PR-AUC"]

fig, ax = plt.subplots(figsize=(11, 7), dpi=150)
bars = ax.bar(display_names, metric_values)
ax.set_title("Final Test Performance — Xception Eye ROI", fontsize=14)
ax.set_ylabel("Score", fontsize=11)
ax.set_ylim(0, 1.05)
ax.tick_params(axis="x", rotation=25)
for bar, value in zip(bars, metric_values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        min(value + 0.02, 1.02),
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=10,
    )
ax.grid(True, axis="y", alpha=0.25)
save_figure(fig, "06_final_metrics.png")

figure_manifest = pd.DataFrame([
    {
        "file": p.name,
        "width_px": Image.open(p).size[0],
        "height_px": Image.open(p).size[1],
    }
    for p in sorted(FIGURES_DIR.glob("*.png"))
])
atomic_csv_dump(figure_manifest, FIGURES_DIR / "figure_manifest.csv")

print(figure_manifest)


                            file  width_px  height_px
0        01_confusion_matrix.png      1129       1002
1               02_roc_curve.png      1485       1035
2  03_precision_recall_curve.png      1485       1035
3              04_loss_curve.png      1484       1035
4           05_roc_auc_curve.png      1485       1035
5           06_final_metrics.png      1635       1035


In [19]:

# ============================================================
# CELL 14 — FINAL AUDIT / SUMMARY / COMPLETION MARKER
# ============================================================

# Final quality gates
assert (OUTPUT_ROOT / "config_resolved.yaml").exists()
assert (OUTPUT_ROOT / "requirements_lock.txt").exists()
assert (CHECKPOINT_DIR / "best.ckpt").exists()
assert (METRICS_DIR / "final_test_metrics.json").exists()
assert (PREDICTIONS_DIR / "test_predictions.csv").exists()
assert len(list(FIGURES_DIR.glob("*.png"))) >= 6

for fig_path in FIGURES_DIR.glob("*.png"):
    with Image.open(fig_path) as im:
        assert min(im.size) >= 600, f"Low-resolution figure: {fig_path}"

final_summary = {
    "run_id": RUN_ID,
    "region": "eye",
    "roi_variant": CONFIG["roi_variant"],
    "model": CONFIG["model_name"],
    "seed": SEED,
    "selected_stage": selected_stage,
    "selected_threshold_from_validation": SELECTED_THRESHOLD,
    "test_metrics": test_metrics,
    "data_accounting": accounting,
    "split_leakage_check": "PASSED",
    "smoke_test": "PASSED",
    "checkpoint_test": "PASSED",
    "inference_test": "PASSED",
    "test_used_for_model_selection": False,
    "output_root": str(OUTPUT_ROOT),
}
atomic_json_dump(final_summary, OUTPUT_ROOT / "experiment_summary.json")

summary_text = f"""
EYE ROI DEEPFAKE DETECTION — XCEPTION
=====================================

Run ID                  : {RUN_ID}
Region                  : Eye
ROI                     : {CONFIG['roi_variant']}
Model                   : {CONFIG['model_name']}
Seed                    : {SEED}
Selected stage          : {selected_stage}
Validation threshold    : {SELECTED_THRESHOLD:.5f}

Test Accuracy           : {test_metrics['accuracy']:.5f}
Test Balanced Accuracy  : {test_metrics['balanced_accuracy']:.5f}
Test Precision          : {test_metrics['precision']:.5f}
Test Recall             : {test_metrics['recall']:.5f}
Test Specificity        : {test_metrics['specificity']:.5f}
Test F1                 : {test_metrics['f1']:.5f}
Test ROC-AUC            : {test_metrics['roc_auc']:.5f}
Test PR-AUC             : {test_metrics['pr_auc']:.5f}

Quality Gates
-------------
Metadata schema         : PASSED
Data accounting         : PASSED
Video split leakage     : PASSED
Exact duplicate leakage : PASSED
Smoke test              : PASSED
Atomic checkpoint       : PASSED
NaN/Inf guards          : ENABLED
Fresh-model inference   : PASSED
Test used for selection : NO

Output
------
{OUTPUT_ROOT}
""".strip()

tmp = OUTPUT_ROOT / "experiment_summary.txt.tmp"
tmp.write_text(summary_text, encoding="utf-8")
os.replace(tmp, OUTPUT_ROOT / "experiment_summary.txt")

# Mark this run as complete. A future notebook execution starts a new run ID.
if ACTIVE_RUN_FILE.exists() and ACTIVE_RUN_FILE.read_text(encoding="utf-8").strip() == RUN_ID:
    ACTIVE_RUN_FILE.unlink()

print(summary_text)
print("\n✓ EXPERIMENT COMPLETED SUCCESSFULLY")


EYE ROI DEEPFAKE DETECTION — XCEPTION

Run ID                  : 20260807_1004_eye_xception_seed42
Region                  : Eye
ROI                     : combined
Model                   : legacy_xception
Seed                    : 42
Selected stage          : finetune
Validation threshold    : 0.30500

Test Accuracy           : 0.57285
Test Balanced Accuracy  : 0.55888
Test Precision          : 0.54839
Test Recall             : 0.98077
Test Specificity        : 0.13699
Test F1                 : 0.70345
Test ROC-AUC            : 0.67698
Test PR-AUC             : 0.68654

Quality Gates
-------------
Metadata schema         : PASSED
Data accounting         : PASSED
Video split leakage     : PASSED
Exact duplicate leakage : PASSED
Smoke test              : PASSED
Atomic checkpoint       : PASSED
NaN/Inf guards          : ENABLED
Fresh-model inference   : PASSED
Test used for selection : NO

Output
------
/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/2